In [ ]:
import serial
import time
from datetime import datetime
import matplotlib.pyplot as plt
from collections import deque
import json

In [ ]:
# !pip install pyserial

In [ ]:
SERIAL_PORT = "COM7" 
BAUD_RATE = 115200
OUTPUT_FILE = "can_log.csv"
TARGET_ID = "0x488"
VALUE_INDEX = (2, 3)  
SCALE = 1 / 4        
MAX_POINTS = 100

In [ ]:
timestamps = deque(maxlen=MAX_POINTS)
values = deque(maxlen=MAX_POINTS)

In [ ]:
# Read and write in csv
def main():
    ser = serial.Serial(SERIAL_PORT, BAUD_RATE, timeout=1)
    print(f"Connected to {SERIAL_PORT} at {BAUD_RATE} baud")
    
    with open(OUTPUT_FILE, "w") as f:
        f.write("timestamp,can_id,data\n")  # CSV header
        
        while True:
            try:
                line = ser.readline().decode("utf-8",  errors='ignore').strip()
                if not line:
                    continue

                # Expected format from ESP: "ID:0x488 DATA:28 15 49 72 A6 00 00 A0"
                if "ID:" in line and "DATA:" in line:
                    parts = line.split()
                    can_id = parts[0].split(":")[1]
                    data = ' '.join(parts[2:])
                    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S.%f")[:-3]
                    f.write(f"{timestamp},{can_id},{data}\n")
                    f.flush()
                    print(f"[{timestamp}] {can_id}: {data}")
            except KeyboardInterrupt:
                print("\nLogging stopped by user.")
                break
            except Exception as e:
                print("Error:", e)

In [ ]:
if __name__ == "__main__":
    main()

In [ ]:
# # only read
# def main():
#     ser = serial.Serial(SERIAL_PORT, BAUD_RATE, timeout=1)
#     print(f"Connected to {SERIAL_PORT} at {BAUD_RATE} baud")
        
#     plt.ion()
#     fig, ax = plt.subplots()
#     line, = ax.plot([], [], label="RPM")
#     ax.set_ylim(0, 6000)
#     ax.set_title(f"Live CAN Signal - {TARGET_ID}")
#     ax.set_xlabel("Samples")
#     ax.set_ylabel("Value")
#     ax.legend()
#     while True:
#         try:
#             line_raw = ser.readline().decode("utf-8").strip()
#             print(line_raw)
#             if not line_raw:
#                 continue
#             if "ID:" in line_raw and "DATA:" in line_raw:
#                 parts = line_raw.split()
#                 can_id = parts[0].split(":")[1]
#                 data = ' '.join(parts[2:])
#                 timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S.%f")[:-3]
#                 print(f"[{timestamp}] {can_id}: {data}")

#             line_raw = line_raw.replace("'", '"').replace(",]", "]")

#             data = json.loads(line_raw)

#             if data.get("id") == TARGET_ID:
#                 data_bytes = data["data"]
#                 if len(data_bytes) >= max(VALUE_INDEX) + 1:
#                     low = int(data_bytes[VALUE_INDEX[0]])
#                     high = int(data_bytes[VALUE_INDEX[1]])
#                     raw = (high << 8) + low
#                     value = raw * SCALE
#                     ax.set_xlim(0, max(len(values), 10))

#                     timestamps.append(datetime.now().strftime("%H:%M:%S"))
#                     values.append(value)

#                     line.set_xdata(range(len(values)))
#                     line.set_ydata(values)
#                     ax.set_xlim(0, len(values))
#                     ax.relim()
#                     ax.autoscale_view()
#                     fig.canvas.draw()
#                     fig.canvas.flush_events()

#                     print(f"[{timestamps[-1]}] {TARGET_ID}: {value:.2f}")
#                     plt.ioff()
#                     plt.show()
#         except KeyboardInterrupt:
#             print("\nLogging stopped by user.")
#             break
#         except Exception as e:
#             print("Error:", e)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib import style

In [ ]:
def print_by_id(id:str, data:pd.DataFrame):
    acelerador = data.loc[data['can_id'] == id]

    acelerador["timestamp"] = pd.to_datetime(acelerador["timestamp"])

    try:
        acelerador[["b0", "b1", "b2", "b3", "b4", "b5", "b6"]] = (
            acelerador["data"]
            .str.split()
            .apply(lambda x: [int(byte) for byte in x])
            .tolist()
        )

        for i in range(7): 
            plt.figure(figsize=(20, 5))
            plt.plot(acelerador["timestamp"], acelerador[f"b{i}"], label=f"Byte {i}")
            plt.xlabel("Time")
            plt.ylabel("Byte Value")
            plt.title("CAN Data Bytes Over Time")
            plt.legend()
            plt.grid(True)
            plt.tight_layout()
            plt.show()

    except Exception as e:
        print(f"Didnt work: {e}")

In [32]:
data = pd.read_csv(r"C:\Users\vinic\Desktop\ESPs3\can_log.csv")

for id in data['can_id'].unique():
    if "320" in id: 
        print_by_id(id, data)
        print(id)
        acelerador = data.loc[data['can_id'] == id]

Didnt work: Columns must be same length as key
0x320


C:\Users\vinic\AppData\Local\Temp\ipykernel_102484\827619743.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  acelerador["timestamp"] = pd.to_datetime(acelerador["timestamp"])


In [33]:
acelerador 

,timestamp,can_id,data
5,2025-06-22 18:08:29.194,0x320,4 0 21 0 0 0 0 0
21,2025-06-22 18:08:29.256,0x320,4 0 21 0 0 0 0 0
26,2025-06-22 18:08:29.277,0x320,4 0 21 0 0 0 0 0
41,2025-06-22 18:08:29.337,0x320,4 0 21 0 0 0 0 0
58,2025-06-22 18:08:29.402,0x320,4 0 21 0 0 0 0 0
...,...,...,...
42344,2025-06-22 18:11:15.339,0x320,4 0 20 0 0 0 0 0
42360,2025-06-22 18:11:15.400,0x320,4 0 20 0 0 0 0 0
42376,2025-06-22 18:11:15.448,0x320,4 0 20 0 0 0 0 0
42382,2025-06-22 18:11:15.481,0x320,4 0 20 0 0 0 0 0


In [ ]:
acelerador[["b0", "b1", "b2", "b3", "b4", "b5", "b6", "b7"]] = (
            acelerador["data"]
            .str.split()
            .apply(lambda x: [int(byte) for byte in x])
            .tolist()
        )

In [ ]:
acelerador

In [ ]:
import serial
import matplotlib.pyplot as plt
import json
from datetime import datetime
from collections import deque

# CONFIG
SERIAL_PORT = "COM7"       # Change to your serial port
BAUD_RATE = 115200
TARGET_ID = "0x488"
BUFFER_SIZE = 100  # How many points to keep in the live window

# Initialize buffers
timestamps = deque(maxlen=BUFFER_SIZE)
byte_values = {f"b{i}": deque(maxlen=BUFFER_SIZE) for i in range(7)}

# Set up real-time plot
plt.ion()
fig, axs = plt.subplots(7, 1, figsize=(20, 14), sharex=True)
fig.subplots_adjust(hspace=0.5)
lines = []

for i in range(7):
    axs[i].set_ylabel(f"Byte {i}")
    axs[i].set_ylim(0, 255)
    axs[i].grid(True)
    line, = axs[i].plot([], [], label=f"Byte {i}")
    axs[i].legend(loc="upper right")
    lines.append(line)

axs[-1].set_xlabel("Samples")

# Connect to serial
ser = serial.Serial(SERIAL_PORT, BAUD_RATE, timeout=1)
print(f"Connected to {SERIAL_PORT} at {BAUD_RATE} baud")

try:
    while True:
        try:
            line_raw = ser.readline().decode("utf-8").strip()
            # print(line_raw)
            if not line_raw:
                continue

            # Pre-clean the string and parse
            line_raw = line_raw.replace("'", '"')
            data = json.loads(line_raw)

            if data.get("id") == TARGET_ID:
                data_bytes = data.get("data", [])
                if len(data_bytes) < 7:
                    continue

                timestamps.append(datetime.now().strftime("%H:%M:%S"))

                for i in range(7):
                    byte_values[f"b{i}"].append(int(data_bytes[i]))

                # Update plots
                for i in range(7):
                    lines[i].set_xdata(range(len(byte_values[f"b{i}"])))
                    lines[i].set_ydata(byte_values[f"b{i}"])
                    axs[i].set_xlim(0, len(byte_values[f"b{i}"]))
                    axs[i].relim()
                    axs[i].autoscale_view()

                fig.canvas.draw()
                fig.canvas.flush_events()

        except json.JSONDecodeError:
            pass
        except Exception as e:
            print("Error:", e)

except KeyboardInterrupt:
    print("\nStopped by user")


In [ ]:
import serial
import matplotlib.pyplot as plt
import json
from datetime import datetime
from collections import deque

# CONFIG
SERIAL_PORT = "COM7"       # Change to your serial port
BAUD_RATE = 115200
TARGET_ID = "0x488"
BUFFER_SIZE = 100 # How many points to keep in the live window

# Initialize buffers
timestamps = deque(maxlen=BUFFER_SIZE)
byte_values = {f"b{i}": deque(maxlen=BUFFER_SIZE) for i in range(7)}

# Set up real-time plot
plt.ion()
fig, axs = plt.subplots(7, 1, figsize=(20, 14), sharex=True)
fig.subplots_adjust(hspace=0.5)
lines = []

for i in range(7):
    axs[i].set_ylabel(f"Byte {i}")
    axs[i].set_ylim(0, 255)
    axs[i].grid(True)
    line, = axs[i].plot([], [], label=f"Byte {i}")
    axs[i].legend(loc="upper right")
    lines.append(line)

axs[-1].set_xlabel("Samples")

# Connect to serial
ser = serial.Serial(SERIAL_PORT, BAUD_RATE, timeout=1)
print(f"Connected to {SERIAL_PORT} at {BAUD_RATE} baud")

try:
    while True:
        try:
            line_raw = ser.readline().decode("utf-8").strip()
            # print(line_raw)
            # if not line_raw:
            #     continue

            # # Pre-clean the string and parse
            # # line_raw = line_raw.replace("'", '"')
            # data = json.loads(line_raw)
            data = line_raw
            # print(data)
            try:
                v0, v1, v2, v3, v4, v5, v6, v7, v8 = data.split(" ")
            except:
                continue
            v1 = v1[4:]
            print(v1)
            values = [v2, v3, v4, v5, v6, v7, v8]
            if TARGET_ID in v0:
                timestamps.append(datetime.now().strftime("%H:%M:%S"))

                if len(values) < 8:
                    continue  # skip invalid data
                for i in range(7):
                    byte_values[f"b{i}"].append(int(values[i]))

                # Update plots
                for i in range(7):
                    lines[i].set_xdata(list(range(len(byte_values[f"b{i}"]))))
                    lines[i].set_ydata([int(v) for v in byte_values[f"b{i}"]])
                    axs[i].set_xlim(0, len(byte_values[f"b{i}"]))
                    axs[i].relim()
                    axs[i].autoscale_view()
                    print("Troleiii")
                plt.pause(1)
                fig.canvas.draw()
                fig.canvas.flush_events()

        except json.JSONDecodeError:
            pass
        except Exception as e:
            print("Error:", e)

except KeyboardInterrupt:
    print("\nStopped by user")
